# 00 Grid Resolution Selection for Karachi Electricity Prediction

This notebook performs **Stage 1: grid-size and preliminary model screening** for four spatial resolutions:

- 1000 m
- 750 m
- 500 m
- 250 m

For every grid size, the notebook rebuilds the same engineered baseline features from the raw data and evaluates three model families:

- Elastic Net
- Random Forest
- XGBoost

The comparison deliberately excludes:

- CNN-derived visual features
- spatial lag of electricity consumption
- final hyperparameter optimisation
- transformation of the modelling target

Validation is kept consistent across all grid sizes:

1. Random 5-fold cross-validation
2. KMeans-based spatial cross-validation

The notebook also reports data-support and sparsity indicators, including electricity-observation counts, POI availability, population, buildings, roads, clipped edge cells, and target variability.

## 1. Imports

Required packages include `geopandas`, `rasterio`, `rasterstats`, `scipy`, `scikit-learn`, and `xgboost`.

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio

from shapely.geometry import box
from shapely import wkt
from rasterstats import zonal_stats
from scipy.spatial import cKDTree
from scipy.stats import entropy

from sklearn.base import clone
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)


## 2. Paths and fixed experiment settings

All grid sizes use the same data sources, feature definitions, model settings, random seed, and spatial-block boundaries.

In [ ]:
# ── Input paths ──────────────────────────────────────────────────────────────
import sys

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from project_config import RAW_DATA_DIR, WORK_DIR

BOUNDARY_PATH = RAW_DATA_DIR / "boundary" / "karachi_boundary_mask.json"
ELEC_PATH = RAW_DATA_DIR / "electricity consumption" / "Electricity_consumption_data.gpkg"
POIS_NEW_PATH = RAW_DATA_DIR / "road" / "Karachi_Spatial_Data_csv" / "pois_new.csv"
ROAD_GPKG = RAW_DATA_DIR / "road" / "Karachi_Spatial_Data.gpkg"

B02_PATH = RAW_DATA_DIR / "daylight21" / "processing" / "B02_k.tif"
B03_PATH = RAW_DATA_DIR / "daylight21" / "processing" / "B03_k.tif"
B04_PATH = RAW_DATA_DIR / "daylight21" / "processing" / "B04_k.tif"
B08_PATH = RAW_DATA_DIR / "daylight21" / "processing" / "B08_k.tif"
B11_PATH = RAW_DATA_DIR / "daylight21" / "processing" / "B11_k.tif"
NTL_PATH = RAW_DATA_DIR / "nightlight" / "VNL_v21_npp_2021_global_vcmslcfg_c202205302300.median_masked_k.tif.tif"
POP_PATH = RAW_DATA_DIR / "population" / "pak_ppp_2020.tif"

BUILD_PATHS = [
    RAW_DATA_DIR / "building density" / "3eb_buildings.csv.gz",
    RAW_DATA_DIR / "building density" / "395_buildings.csv.gz",
]
BUILDING_CLIPPED_PATH = RAW_DATA_DIR / "building density" / "karachi_buildings.gpkg"

# Notebook 01 reads the selected 500 m files from this same directory.
OUTPUT_ROOT = WORK_DIR / "grid_size_selection"

GRID_OUTPUT_DIR = OUTPUT_ROOT / "grids"
FEATURE_OUTPUT_DIR = OUTPUT_ROOT / "features"
RESULT_OUTPUT_DIR = OUTPUT_ROOT / "results"

for folder in [OUTPUT_ROOT, GRID_OUTPUT_DIR, FEATURE_OUTPUT_DIR, RESULT_OUTPUT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Grid-resolution outputs:", OUTPUT_ROOT)

# ── Fixed experiment settings ────────────────────────────────────────────────
PROJECT_CRS = "EPSG:32642"
GRID_SIZES = [1000, 750, 500, 250]
RANDOM_STATE = 42
RANDOM_FOLDS = 5
SPATIAL_BLOCKS_PER_AXIS = 3
MIN_SPATIAL_TEST_SAMPLES = 5

ELEC_VALUE_COLUMN = "Avg_Avg_Cs"
TARGET_COLUMN = "elec_consumption"

BAND_FILES = {
    "B02": B02_PATH,
    "B03": B03_PATH,
    "B04": B04_PATH,
    "B08": B08_PATH,
    "B11": B11_PATH,
}


In [ ]:
# Validate all required input files before starting the long workflow.
required_files = [
    BOUNDARY_PATH, ELEC_PATH, POIS_NEW_PATH, ROAD_GPKG,
    B02_PATH, B03_PATH, B04_PATH, B08_PATH, B11_PATH,
    NTL_PATH, POP_PATH,
]

# Raw building tiles are only required when the cached clipped building file
# does not already exist.
if not os.path.exists(BUILDING_CLIPPED_PATH):
    required_files.extend(BUILD_PATHS)

missing_files = [path for path in required_files if not os.path.exists(path)]

if missing_files:
    raise FileNotFoundError(
        "The following required files were not found:\n"
        + "\n".join(missing_files)
    )

print("All required input files were found.")
print("Outputs will be saved under:", OUTPUT_ROOT)


## 3. Load source datasets once

The large source layers are loaded and projected once, then reused for all four grid sizes.  
The same projected boundary is also used to define fixed 3 × 3 spatial blocks, so the geographical validation regions do not move between grid sizes.

In [ ]:
# Boundary
boundary = gpd.read_file(BOUNDARY_PATH).to_crs(PROJECT_CRS)
boundary = boundary.dissolve()[["geometry"]]
boundary_minx, boundary_miny, boundary_maxx, boundary_maxy = boundary.total_bounds

# Electricity points
electricity = gpd.read_file(ELEC_PATH).to_crs(PROJECT_CRS)
if ELEC_VALUE_COLUMN not in electricity.columns:
    raise KeyError(
        f"Electricity column '{ELEC_VALUE_COLUMN}' was not found. "
        f"Available columns: {electricity.columns.tolist()}"
    )
electricity = electricity[
    electricity.geometry.notna()
    & electricity[ELEC_VALUE_COLUMN].notna()
][[ELEC_VALUE_COLUMN, "geometry"]].copy()

# POIs
pois_raw = pd.read_csv(POIS_NEW_PATH)
required_poi_cols = {"new_category", "geometry"}
if not required_poi_cols.issubset(pois_raw.columns):
    raise KeyError(
        f"POI file must contain {required_poi_cols}. "
        f"Available columns: {pois_raw.columns.tolist()}"
    )

pois_raw["new_category"] = (
    pois_raw["new_category"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({"others": "other"})
)

def parse_wkt_safely(value):
    try:
        return wkt.loads(value)
    except Exception:
        return None

pois_raw["geometry"] = pois_raw["geometry"].apply(parse_wkt_safely)
pois_raw = pois_raw[pois_raw["geometry"].notna()].copy()

sample_geom = pois_raw["geometry"].iloc[0]
if -180 <= sample_geom.x <= 180 and -90 <= sample_geom.y <= 90:
    pois = gpd.GeoDataFrame(
        pois_raw,
        geometry="geometry",
        crs="EPSG:4326"
    ).to_crs(PROJECT_CRS)
else:
    pois = gpd.GeoDataFrame(
        pois_raw,
        geometry="geometry",
        crs=PROJECT_CRS
    )

# Roads
roads = gpd.read_file(ROAD_GPKG, layer="roads").to_crs(PROJECT_CRS)
roads = roads[roads.geometry.notna()].copy()

major_types = {
    "motorway", "trunk", "primary", "secondary",
    "motorway_link", "trunk_link", "primary_link", "secondary_link"
}
roads["is_major"] = roads["highway"].isin(major_types)
major_roads = roads[roads["is_major"]].copy()

if len(major_roads) > 0:
    major_road_union = (
        major_roads.geometry.union_all()
        if hasattr(major_roads.geometry, "union_all")
        else major_roads.geometry.unary_union
    )
else:
    major_road_union = None

print("Boundary bounds:", boundary.total_bounds.round(2))
print("Electricity points:", len(electricity))
print("POIs:", len(pois))
print("Road features:", len(roads))
print("Major road features:", len(major_roads))


In [ ]:
# Load or create the clipped Karachi building-footprint layer once.
if os.path.exists(BUILDING_CLIPPED_PATH):
    print("Loading cached Karachi building footprints...")
    buildings_karachi = gpd.read_file(BUILDING_CLIPPED_PATH)
else:
    print("Creating cached Karachi building footprints from raw CSV tiles...")
    boundary_wgs84 = boundary.to_crs("EPSG:4326")
    minx, miny, maxx, maxy = boundary_wgs84.total_bounds

    building_parts = []
    for path in BUILD_PATHS:
        print("Loading:", os.path.basename(path))
        part = pd.read_csv(path, compression="gzip")
        required_build_cols = {"longitude", "latitude", "geometry", "area_in_meters"}
        if not required_build_cols.issubset(part.columns):
            raise KeyError(
                f"{path} must contain {required_build_cols}. "
                f"Available columns: {part.columns.tolist()}"
            )

        part = part[
            (part["longitude"] >= minx)
            & (part["longitude"] <= maxx)
            & (part["latitude"] >= miny)
            & (part["latitude"] <= maxy)
        ].copy()
        building_parts.append(part)

    buildings_df = pd.concat(building_parts, ignore_index=True)
    buildings_df["geometry"] = buildings_df["geometry"].apply(parse_wkt_safely)
    buildings_df = buildings_df[buildings_df["geometry"].notna()].copy()

    buildings_gdf = gpd.GeoDataFrame(
        buildings_df,
        geometry="geometry",
        crs="EPSG:4326"
    )
    buildings_karachi = gpd.clip(buildings_gdf, boundary_wgs84)
    buildings_karachi.to_file(BUILDING_CLIPPED_PATH, driver="GPKG")
    print("Saved:", BUILDING_CLIPPED_PATH)

buildings_utm = buildings_karachi.to_crs(PROJECT_CRS)

if "area_in_meters" not in buildings_utm.columns:
    # Fallback only when the cached file does not retain the original area field.
    buildings_utm["area_in_meters"] = buildings_utm.geometry.area

building_points = buildings_utm[["area_in_meters", "geometry"]].copy()
building_points["geometry"] = building_points.geometry.representative_point()

print("Karachi building footprints:", len(buildings_utm))


## 4. Shared feature-engineering functions

Each resolution receives the same baseline feature families:

- Sentinel-2 summary statistics and spectral indices
- VIIRS nighttime-light statistics
- road length, road density, major-road share, and distance to a major road
- POI category counts, total count, Shannon diversity, and nearest distances
- WorldPop total population and population density
- building count, footprint-area statistics, and building coverage

`dist_other_poi` and `pop_pixel_count` are excluded, matching the cleaned 500 m feature matrix.

In [ ]:
# ============================================================
# 42 candidate baseline predictors for grid-size screening
# ============================================================

BASELINE_FEATURES = [
    "area_m2",

    "B02_mean", "B02_std", "B02_max",
    "B03_mean", "B03_std", "B03_max",
    "B04_mean", "B04_std", "B04_max",
    "B08_mean", "B08_std", "B08_max",
    "B11_mean", "B11_std", "B11_max",

    "NDVI", "NDBI", "brightness",

    "NTL_mean", "NTL_std", "NTL_max",

    "dist_major_road",
    "road_length_total",
    "road_count",
    "road_length_major",
    "road_density",
    "major_road_ratio",

    "poi_economic_count",
    "poi_social_count",
    "poi_other_count",
    "poi_total_count",
    "poi_shannon",
    "dist_economic_poi",
    "dist_social_poi",

    "pop_total",
    "pop_density_km2",

    "building_count",
    "building_area_total",
    "building_area_mean",
    "building_area_std",
    "building_coverage",
]

META_AND_TARGET_COLUMNS = [
    "grid_id",
    "centroid_x",
    "centroid_y",
    "elec_consumption",
    "elec_point_count",
    "elec_std",
]

assert len(BASELINE_FEATURES) == 42

print("Candidate baseline feature count:", len(BASELINE_FEATURES))

In [ ]:
def add_raster_stats(
    gdf,
    raster_path,
    prefix,
    stats=("mean", "std", "max"),
    nodata_fallback=-1,
):
    """Add raster zonal statistics while respecting the raster's own CRS."""
    with rasterio.open(raster_path) as src:
        raster_crs = src.crs
        nodata = src.nodata if src.nodata is not None else nodata_fallback

    zones = gdf.to_crs(raster_crs)
    values = zonal_stats(
        zones,
        raster_path,
        stats=list(stats),
        nodata=nodata,
    )

    for stat in stats:
        gdf[f"{prefix}_{stat}"] = [item.get(stat) for item in values]

    return gdf


def shannon_from_counts(row, columns):
    counts = row[columns].to_numpy(dtype=float)
    counts = counts[counts > 0]

    if len(counts) == 0:
        return 0.0

    probabilities = counts / counts.sum()
    return float(entropy(probabilities, base=np.e))



def nearest_point_distance(grid_gdf, point_gdf):
    """Distance from each grid centroid to the nearest point feature."""
    if len(point_gdf) == 0:
        return np.full(len(grid_gdf), np.nan)

    grid_coords = np.column_stack([
        grid_gdf["centroid_x"].to_numpy(),
        grid_gdf["centroid_y"].to_numpy(),
    ])
    point_coords = np.column_stack([
        point_gdf.geometry.x.to_numpy(),
        point_gdf.geometry.y.to_numpy(),
    ])

    tree = cKDTree(point_coords)
    distances, _ = tree.query(grid_coords, k=1)
    return distances


In [ ]:
def create_clipped_grid(cell_size):
    """Create a boundary-clipped grid using a common origin for all scales."""
    xs = np.arange(boundary_minx, boundary_maxx, cell_size)
    ys = np.arange(boundary_miny, boundary_maxy, cell_size)

    cells = [
        box(x, y, x + cell_size, y + cell_size)
        for x in xs
        for y in ys
    ]

    full_grid = gpd.GeoDataFrame(
        {"geometry": cells},
        crs=PROJECT_CRS,
    )

    clipped_grid = gpd.overlay(
        full_grid,
        boundary,
        how="intersection",
    ).reset_index(drop=True)

    clipped_grid["grid_id"] = np.arange(len(clipped_grid))
    clipped_grid["area_m2"] = clipped_grid.geometry.area
    clipped_grid["centroid_x"] = clipped_grid.geometry.centroid.x
    clipped_grid["centroid_y"] = clipped_grid.geometry.centroid.y

    return clipped_grid


In [ ]:
def build_feature_matrix(cell_size):
    """
    Rebuild one complete labelled-grid feature matrix from the raw sources.

    The returned model matrix contains no CNN features and no spatial lag.
    """
    print("\n" + "=" * 90)
    print(f"BUILDING {cell_size} m GRID")
    print("=" * 90)

    expected_area = float(cell_size ** 2)

    # ── A. Grid generation ──────────────────────────────────────────────────
    all_grid = create_clipped_grid(cell_size)
    all_grid_count = len(all_grid)
    all_edge_cell_pct = (
        (all_grid["area_m2"] < expected_area * 0.99).mean() * 100
    )

    # ── B. Electricity aggregation ──────────────────────────────────────────
    joined_electricity = gpd.sjoin(
        electricity,
        all_grid[["grid_id", "geometry"]],
        how="inner",
        predicate="within",
    )

    electricity_agg = (
        joined_electricity
        .groupby("grid_id")
        .agg(
            elec_consumption=(ELEC_VALUE_COLUMN, "mean"),
            elec_point_count=(ELEC_VALUE_COLUMN, "count"),
            elec_std=(ELEC_VALUE_COLUMN, "std"),
        )
        .reset_index()
    )

    grid = all_grid.merge(
        electricity_agg,
        on="grid_id",
        how="left",
    )

    # The modelling sample contains only cells with observed electricity.
    grid = grid[grid[TARGET_COLUMN].notna()].copy().reset_index(drop=True)
    grid["grid_id"] = np.arange(len(grid))
    grid["elec_std"] = grid["elec_std"].fillna(0)

    labelled_grid_count = len(grid)
    labelled_grid_pct = labelled_grid_count / all_grid_count * 100
    labelled_edge_cell_pct = (
        (grid["area_m2"] < expected_area * 0.99).mean() * 100
    )

    # ── C. POI counts ───────────────────────────────────────────────────────
    joined_pois = gpd.sjoin(
        pois[["new_category", "geometry"]],
        grid[["grid_id", "geometry"]],
        how="inner",
        predicate="within",
    )

    poi_counts = (
        joined_pois
        .groupby(["grid_id", "new_category"])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    poi_counts = poi_counts.rename(columns={
        "economic": "poi_economic_count",
        "social": "poi_social_count",
        "other": "poi_other_count",
    })

    grid = grid.merge(poi_counts, on="grid_id", how="left")

    poi_count_cols = [
        "poi_economic_count",
        "poi_social_count",
        "poi_other_count",
    ]

    for column in poi_count_cols:
        if column not in grid.columns:
            grid[column] = 0
        grid[column] = grid[column].fillna(0).astype(int)

    grid["poi_total_count"] = grid[poi_count_cols].sum(axis=1)
    grid["poi_shannon"] = grid.apply(
        shannon_from_counts,
        axis=1,
        columns=poi_count_cols,
    )

    grid["dist_economic_poi"] = nearest_point_distance(
        grid,
        pois[pois["new_category"] == "economic"],
    )
    grid["dist_social_poi"] = nearest_point_distance(
        grid,
        pois[pois["new_category"] == "social"],
    )

    # ── D. Sentinel-2 and VIIRS ─────────────────────────────────────────────
    for band_name, band_path in BAND_FILES.items():
        print(f"  Raster statistics: {band_name}")
        grid = add_raster_stats(grid, band_path, band_name)

    print("  Raster statistics: NTL")
    grid = add_raster_stats(grid, NTL_PATH, "NTL")

    grid["NDVI"] = (
        (grid["B08_mean"] - grid["B04_mean"])
        / (grid["B08_mean"] + grid["B04_mean"] + 1e-9)
    )
    grid["NDBI"] = (
        (grid["B11_mean"] - grid["B08_mean"])
        / (grid["B11_mean"] + grid["B08_mean"] + 1e-9)
    )
    grid["brightness"] = (
        grid["B02_mean"]
        + grid["B03_mean"]
        + grid["B04_mean"]
        + grid["B08_mean"]
    ) / 4

    # ── E. Roads ────────────────────────────────────────────────────────────
    print("  Road overlay")
    roads_clipped = gpd.overlay(
        roads[["highway", "is_major", "geometry"]],
        grid[["grid_id", "geometry"]],
        how="intersection",
    )

    if len(roads_clipped) > 0:
        roads_clipped["length_m"] = roads_clipped.geometry.length

        road_agg = (
            roads_clipped
            .groupby("grid_id")
            .agg(
                road_length_total=("length_m", "sum"),
                road_count=("length_m", "count"),
            )
            .reset_index()
        )

        major_agg = (
            roads_clipped[roads_clipped["is_major"]]
            .groupby("grid_id")["length_m"]
            .sum()
            .reset_index(name="road_length_major")
        )

        road_agg = road_agg.merge(
            major_agg,
            on="grid_id",
            how="left",
        )
    else:
        road_agg = pd.DataFrame({
            "grid_id": grid["grid_id"],
            "road_length_total": 0.0,
            "road_count": 0,
            "road_length_major": 0.0,
        })

    road_agg["road_length_major"] = road_agg[
        "road_length_major"
    ].fillna(0)

    grid = grid.merge(
        road_agg,
        on="grid_id",
        how="left",
    )

    road_count_and_length_cols = [
        "road_length_total",
        "road_count",
        "road_length_major",
    ]
    grid[road_count_and_length_cols] = grid[
        road_count_and_length_cols
    ].fillna(0)

    grid["road_density"] = (
        (grid["road_length_total"] / 1000)
        / (grid["area_m2"] / 1_000_000)
    )
    grid["major_road_ratio"] = (
        grid["road_length_major"]
        / (grid["road_length_total"] + 1e-9)
    )

    if major_road_union is not None:
        grid["dist_major_road"] = grid.geometry.centroid.distance(
            major_road_union
        )
    else:
        grid["dist_major_road"] = np.nan

    # ── F. Population ───────────────────────────────────────────────────────
    print("  Population statistics")
    with rasterio.open(POP_PATH) as population_src:
        population_crs = population_src.crs
        population_nodata = (
            population_src.nodata
            if population_src.nodata is not None
            else -99999
        )

    population_stats = zonal_stats(
        grid.to_crs(population_crs),
        POP_PATH,
        stats=["sum"],
        nodata=population_nodata,
    )

    grid["pop_total"] = [
        item["sum"] if item["sum"] is not None else 0
        for item in population_stats
    ]
    grid["pop_density_km2"] = (
        grid["pop_total"]
        / (grid["area_m2"] / 1_000_000)
    )

    # ── G. Buildings ────────────────────────────────────────────────────────
    print("  Building aggregation")
    joined_buildings = gpd.sjoin(
        building_points,
        grid[["grid_id", "geometry"]],
        how="inner",
        predicate="within",
    )

    building_agg = (
        joined_buildings
        .groupby("grid_id")
        .agg(
            building_count=("area_in_meters", "count"),
            building_area_total=("area_in_meters", "sum"),
            building_area_mean=("area_in_meters", "mean"),
            building_area_std=("area_in_meters", "std"),
        )
        .reset_index()
    )

    grid = grid.merge(
        building_agg,
        on="grid_id",
        how="left",
    )

    building_cols = [
        "building_count",
        "building_area_total",
        "building_area_mean",
        "building_area_std",
    ]
    grid[building_cols] = grid[building_cols].fillna(0)
    grid["building_coverage"] = (
        grid["building_area_total"]
        / (grid["area_m2"] + 1e-9)
    )


    # ── H. Build the identical model matrix ─────────────────────────────────
    missing_features = [
        column
        for column in BASELINE_FEATURES
        if column not in grid.columns
    ]
    if missing_features:
        raise KeyError(
            f"{cell_size} m grid is missing baseline features: "
            f"{missing_features}"
        )

    model_columns = META_AND_TARGET_COLUMNS + BASELINE_FEATURES
    matrix = grid[model_columns].copy()

    # Exact same column order for every resolution.
    matrix = matrix[META_AND_TARGET_COLUMNS + BASELINE_FEATURES]

    # Save per-resolution outputs.
    feature_path = FEATURE_OUTPUT_DIR / f"feature_matrix_{cell_size}m.csv"
    grid_path = GRID_OUTPUT_DIR / f"grid_features_{cell_size}m.gpkg"

    matrix.to_csv(feature_path, index=False)
    grid.to_file(grid_path, driver="GPKG")

    build_info = {
        "grid_size_m": cell_size,
        "expected_full_area_m2": expected_area,
        "all_boundary_grid_count": all_grid_count,
        "labelled_grid_count": labelled_grid_count,
        "labelled_grid_pct": labelled_grid_pct,
        "all_edge_cell_pct": all_edge_cell_pct,
        "labelled_edge_cell_pct": labelled_edge_cell_pct,
        "feature_matrix_path": str(feature_path),
        "feature_grid_path": str(grid_path),
    }

    print(f"  Labelled cells: {labelled_grid_count:,}")
    print(f"  Feature matrix: {feature_path}")
    print(f"  Feature grid:   {grid_path}")

    return matrix, grid, build_info


## 5. Rebuild all four grid sizes

This is the most computationally expensive section. The 250 m grid has the largest number of cells, so its raster and vector operations will take the longest.

In [ ]:
matrices_by_size = {}
grids_by_size = {}
build_information = []

for grid_size in GRID_SIZES:
    matrix, feature_grid, build_info = build_feature_matrix(grid_size)

    matrices_by_size[grid_size] = matrix
    grids_by_size[grid_size] = feature_grid
    build_information.append(build_info)

build_information = pd.DataFrame(build_information).sort_values("grid_size_m")
display(build_information)


## 6. Data-richness and sparsity indicators

Raw counts naturally rise with grid area, so the notebook reports both counts and unit-area densities.  
These indicators are supporting evidence rather than the sole grid-selection criterion.

In [ ]:
def percentage_true(series):
    return float(series.fillna(False).mean() * 100)


def coefficient_of_variation(series):
    mean_value = series.mean()
    if mean_value == 0 or pd.isna(mean_value):
        return np.nan
    return float(series.std(ddof=0) / mean_value)


def compute_richness_metrics(cell_size, matrix, grid, build_info):
    area_km2 = matrix["area_m2"] / 1_000_000
    poi_density_km2 = matrix["poi_total_count"] / area_km2

    target_q25 = matrix[TARGET_COLUMN].quantile(0.25)
    target_q75 = matrix[TARGET_COLUMN].quantile(0.75)

    return {
        "grid_size_m": cell_size,
        "all_boundary_grid_count": build_info["all_boundary_grid_count"],
        "labelled_grid_count": len(matrix),
        "labelled_grid_pct": build_info["labelled_grid_pct"],
        "all_edge_cell_pct": build_info["all_edge_cell_pct"],
        "labelled_edge_cell_pct": build_info["labelled_edge_cell_pct"],

        "elec_points_total": matrix["elec_point_count"].sum(),
        "elec_points_mean": matrix["elec_point_count"].mean(),
        "elec_points_median": matrix["elec_point_count"].median(),
        "elec_points_p10": matrix["elec_point_count"].quantile(0.10),
        "elec_points_p25": matrix["elec_point_count"].quantile(0.25),
        "elec_points_min": matrix["elec_point_count"].min(),
        "elec_points_max": matrix["elec_point_count"].max(),
        "elec_points_lt5_pct": percentage_true(
            matrix["elec_point_count"] < 5
        ),
        "elec_points_lt10_pct": percentage_true(
            matrix["elec_point_count"] < 10
        ),

        "poi_total_mean": matrix["poi_total_count"].mean(),
        "poi_total_median": matrix["poi_total_count"].median(),
        "poi_total_zero_pct": percentage_true(
            matrix["poi_total_count"] == 0
        ),
        "poi_density_km2_mean": poi_density_km2.mean(),
        "poi_density_km2_median": poi_density_km2.median(),
        "poi_economic_zero_pct": percentage_true(
            matrix["poi_economic_count"] == 0
        ),
        "poi_social_zero_pct": percentage_true(
            matrix["poi_social_count"] == 0
        ),
        "poi_other_zero_pct": percentage_true(
            matrix["poi_other_count"] == 0
        ),

        "population_zero_pct": percentage_true(
            matrix["pop_total"] == 0
        ),
        "building_zero_pct": percentage_true(
            matrix["building_count"] == 0
        ),
        "road_zero_pct": percentage_true(
            matrix["road_count"] == 0
        ),

        "road_density_mean": matrix["road_density"].mean(),
        "building_coverage_mean": matrix["building_coverage"].mean(),
        "population_density_mean": matrix["pop_density_km2"].mean(),

        "target_mean": matrix[TARGET_COLUMN].mean(),
        "target_std": matrix[TARGET_COLUMN].std(ddof=0),
        "target_cv": coefficient_of_variation(matrix[TARGET_COLUMN]),
        "target_min": matrix[TARGET_COLUMN].min(),
        "target_q25": target_q25,
        "target_median": matrix[TARGET_COLUMN].median(),
        "target_q75": target_q75,
        "target_iqr": target_q75 - target_q25,
        "target_max": matrix[TARGET_COLUMN].max(),
    }


build_info_lookup = {
    int(row["grid_size_m"]): row.to_dict()
    for _, row in build_information.iterrows()
}

richness_rows = []

for grid_size in GRID_SIZES:
    richness_rows.append(
        compute_richness_metrics(
            grid_size,
            matrices_by_size[grid_size],
            grids_by_size[grid_size],
            build_info_lookup[grid_size],
        )
    )

richness_summary = (
    pd.DataFrame(richness_rows)
    .sort_values("grid_size_m")
    .reset_index(drop=True)
)

richness_path = RESULT_OUTPUT_DIR / "grid_data_richness_summary.csv"
richness_summary.to_csv(richness_path, index=False)

print("Saved:", richness_path)
display(richness_summary.round(3))


In [ ]:
# Compact views for interpretation
electricity_richness = richness_summary[[
    "grid_size_m",
    "labelled_grid_count",
    "labelled_grid_pct",
    "elec_points_mean",
    "elec_points_median",
    "elec_points_lt5_pct",
    "elec_points_lt10_pct",
    "target_std",
    "target_cv",
    "target_iqr",
]].copy()

urban_feature_richness = richness_summary[[
    "grid_size_m",
    "poi_total_mean",
    "poi_total_median",
    "poi_total_zero_pct",
    "poi_density_km2_mean",
    "population_zero_pct",
    "building_zero_pct",
    "road_zero_pct",
    "labelled_edge_cell_pct",
]].copy()

print("Electricity-label richness and target variability")
display(electricity_richness.round(3))

print("Urban-feature richness and sparsity")
display(urban_feature_richness.round(3))


In [ ]:
# Data-richness plots
plot_data = richness_summary.sort_values("grid_size_m")

plt.figure(figsize=(8, 5))
plt.plot(
    plot_data["grid_size_m"],
    plot_data["elec_points_median"],
    marker="o",
)
plt.xlabel("Grid size (m)")
plt.ylabel("Median electricity points per labelled grid")
plt.title("Electricity-point richness by grid size")
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(
    plot_data["grid_size_m"],
    plot_data["elec_points_lt5_pct"],
    marker="o",
    label="< 5 electricity points",
)
plt.plot(
    plot_data["grid_size_m"],
    plot_data["poi_total_zero_pct"],
    marker="o",
    label="Zero POIs",
)
plt.plot(
    plot_data["grid_size_m"],
    plot_data["building_zero_pct"],
    marker="o",
    label="Zero buildings",
)
plt.xlabel("Grid size (m)")
plt.ylabel("Percentage of labelled grids")
plt.title("Grid sparsity indicators")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(
    plot_data["grid_size_m"],
    plot_data["target_cv"],
    marker="o",
)
plt.xlabel("Grid size (m)")
plt.ylabel("Target coefficient of variation")
plt.title("Electricity-consumption variability retained at each scale")
plt.grid(alpha=0.3)
plt.show()


## 7. Fixed baseline models

Three model families: Elastic Net (linear, L1+L2 regularised), Random Forest, XGBoost.
Elastic Net replaces separate Ridge/Lasso runs — its `l1_ratio` is tuned via internal CV,
so it covers both penalty types in a single model.

In [ ]:
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler

def make_models():
    return {
        "ElasticNet": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "model",
                ElasticNetCV(
                    l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
                    alphas=None,
                    cv=5,
                    random_state=RANDOM_STATE,
                    max_iter=10000,
                    n_jobs=-1,
                ),
            ),
        ]),
        "RandomForest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                RandomForestRegressor(
                    n_estimators=300,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            ),
        ]),
        "XGBoost": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            (
                "model",
                XGBRegressor(
                    n_estimators=500,
                    learning_rate=0.03,
                    max_depth=6,
                    objective="reg:squarederror",
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                    verbosity=0,
                ),
            ),
        ]),
    }

print("Models:", list(make_models().keys()))

## 8. Spatial validation via KMeans

A single reference KMeans model is fit once on the finest resolution (250 m) centroids —
the resolution with the most complete spatial coverage — and every other resolution's
centroids are assigned to the nearest reference cluster via `.predict()`. This keeps the
geographic regions used for validation identical across grid sizes, so the grid-size
comparison in Section 11 measures resolution effects, not differences in fold geometry.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


silhouette_results = []

for grid_size in GRID_SIZES:
    matrix = matrices_by_size[grid_size]
    coords = matrix[["centroid_x", "centroid_y"]].to_numpy()

    for k in range(3, 11):
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        labels = km.fit_predict(coords)
        score = silhouette_score(coords, labels)

        silhouette_results.append({
            "grid_size_m": grid_size,
            "k": k,
            "silhouette_score": score,
        })

silhouette_df = pd.DataFrame(silhouette_results)

fig, ax = plt.subplots(figsize=(8, 5))
for grid_size in GRID_SIZES:
    subset = silhouette_df[silhouette_df["grid_size_m"] == grid_size]
    ax.plot(subset["k"], subset["silhouette_score"], marker="o", label=f"{grid_size}m")

ax.set_xlabel("K (number of clusters)")
ax.set_ylabel("Silhouette score")
ax.set_title("Clustering quality vs K (independent of downstream model)")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
from sklearn.cluster import KMeans

elbow_results = []

for grid_size in GRID_SIZES:
    matrix = matrices_by_size[grid_size]
    coords = matrix[["centroid_x", "centroid_y"]].to_numpy()

    for k in range(3, 11):
        km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
        km.fit(coords)
        elbow_results.append({
            "grid_size_m": grid_size,
            "k": k,
            "inertia": km.inertia_,
        })

elbow_df = pd.DataFrame(elbow_results)

fig, ax = plt.subplots(figsize=(8, 5))
for grid_size in GRID_SIZES:
    subset = elbow_df[elbow_df["grid_size_m"] == grid_size]
    ax.plot(subset["k"], subset["inertia"], marker="o", label=f"{grid_size}m")

ax.set_xlabel("K (number of clusters)")
ax.set_ylabel("Inertia (within-cluster sum of squares)")
ax.set_title("Elbow method: inertia vs K")
ax.legend()
ax.grid(alpha=0.3)
plt.show()

In [ ]:
def cluster_balance_check(matrix, k, random_state=42):
    coords = matrix[["centroid_x", "centroid_y"]].to_numpy()
    km = KMeans(n_clusters=k, random_state=random_state, n_init=10)
    labels = km.fit_predict(coords)
    counts = pd.Series(labels).value_counts()
    return {
        "k": k,
        "min_cluster_size": counts.min(),
        "max_cluster_size": counts.max(),
        "balance_ratio": counts.min() / counts.max(),  # 越接近1越均衡
    }

balance_results = pd.DataFrame([
    cluster_balance_check(matrices_by_size[500], k)
    for k in range(3, 11)
])
display(balance_results)

In [ ]:
from sklearn.cluster import KMeans

N_SPATIAL_CLUSTERS = 4
REFERENCE_GRID_SIZE = min(GRID_SIZES)  # 250 m: finest, most complete spatial coverage

reference_coords = matrices_by_size[REFERENCE_GRID_SIZE][
    ["centroid_x", "centroid_y"]
].to_numpy()

reference_kmeans = KMeans(
    n_clusters=N_SPATIAL_CLUSTERS,
    random_state=RANDOM_STATE,
    n_init=10,
).fit(reference_coords)

print(f"Reference KMeans fit on {REFERENCE_GRID_SIZE} m centroids "
      f"({len(reference_coords)} points, {N_SPATIAL_CLUSTERS} clusters)")


def assign_kmeans_blocks(matrix):
    """Assign each grid to the nearest reference-KMeans cluster (shared across sizes)."""
    coords = matrix[["centroid_x", "centroid_y"]].to_numpy()
    return pd.Series(
        reference_kmeans.predict(coords),
        index=matrix.index,
        dtype="Int64",
    )

In [ ]:
import seaborn as sns
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

# ── Shared colour palette for all cluster/block visualisations ──────────────
SPATIAL_PALETTE = sns.color_palette("Set2", n_colors=N_SPATIAL_CLUSTERS)
SPATIAL_CMAP = mcolors.ListedColormap(SPATIAL_PALETTE)

fig, axes = plt.subplots(1, len(GRID_SIZES), figsize=(5 * len(GRID_SIZES), 5), sharex=True, sharey=True)

for ax, grid_size in zip(axes, sorted(GRID_SIZES, reverse=True)):
    grid_gdf = grids_by_size[grid_size].copy()
    grid_gdf["spatial_block"] = assign_kmeans_blocks(grid_gdf).astype(int)

    grid_gdf.plot(
        column="spatial_block",
        cmap=SPATIAL_CMAP,
        vmin=0,
        vmax=N_SPATIAL_CLUSTERS - 1,
        edgecolor="white",
        linewidth=0.3,
        ax=ax,
    )
    ax.set_title(f"{grid_size} m  (n={len(grid_gdf)})")
    ax.set_aspect("equal")
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel("")
    ax.set_ylabel("")

# Shared legend using the same palette/order
legend_handles = [
    mpatches.Patch(color=SPATIAL_PALETTE[i], label=f"Cluster {i}")
    for i in range(N_SPATIAL_CLUSTERS)
]
fig.legend(
    handles=legend_handles,
    title="Spatial fold",
    loc="lower center",
    ncol=N_SPATIAL_CLUSTERS,
    bbox_to_anchor=(0.5, -0.05),
)

fig.suptitle("KMeans spatial folds (reference fit on 250 m centroids)", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
def spatial_block_diagnostics(cell_size, matrix):
    diagnostics = matrix[["centroid_x", "centroid_y", TARGET_COLUMN]].copy()
    diagnostics["spatial_block"] = assign_kmeans_blocks(matrix)

    return (
        diagnostics
        .groupby("spatial_block", dropna=False)
        .agg(
            n_samples=(TARGET_COLUMN, "size"),
            target_mean=(TARGET_COLUMN, "mean"),
            target_std=(TARGET_COLUMN, "std"),
            target_min=(TARGET_COLUMN, "min"),
            target_max=(TARGET_COLUMN, "max"),
        )
        .reset_index()
        .assign(grid_size_m=cell_size)
    )


block_diagnostics = pd.concat(
    [spatial_block_diagnostics(gs, matrices_by_size[gs]) for gs in GRID_SIZES],
    ignore_index=True,
)

block_diagnostics_path = RESULT_OUTPUT_DIR / "spatial_kmeans_block_diagnostics.csv"
block_diagnostics.to_csv(block_diagnostics_path, index=False)

print("Saved:", block_diagnostics_path)
display(block_diagnostics.round(3))

## 9. Shared validation functions

- Random validation: 5 shuffled folds, seed 42 (unchanged).
- Spatial validation: leave-one-KMeans-cluster-out, using the shared reference clustering
  from Section 8. Produces up to `N_SPATIAL_CLUSTERS` folds per grid size.

In [ ]:
def calculate_metrics(y_true, predictions):
    return {
        "R2": r2_score(y_true, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_true, predictions)),
        "MAE": mean_absolute_error(y_true, predictions),
    }


def summarise_fold_results(fold_results):
    return (
        fold_results
        .groupby(["grid_size_m", "validation", "model"], as_index=False)
        .agg(
            n_folds=("fold", "count"),
            n_samples=("n_test", "sum"),
            R2_mean=("R2", "mean"),
            R2_std=("R2", lambda v: v.std(ddof=0)),
            RMSE_mean=("RMSE", "mean"),
            RMSE_std=("RMSE", lambda v: v.std(ddof=0)),
            MAE_mean=("MAE", "mean"),
            MAE_std=("MAE", lambda v: v.std(ddof=0)),
            train_R2_mean=("train_R2", "mean"),
        )
    )


def evaluate_random_5fold(cell_size, matrix):
    X = matrix[BASELINE_FEATURES].copy()
    y = matrix[TARGET_COLUMN].copy()

    splitter = KFold(n_splits=RANDOM_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    split_indices = list(splitter.split(X))
    rows = []

    for model_name, model_template in make_models().items():
        print(f"{cell_size} m | random 5-fold | {model_name}")
        for fold_number, (train_index, test_index) in enumerate(split_indices, start=1):
            model = clone(model_template)
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train, y_test = y.iloc[train_index], y.iloc[test_index]

            model.fit(X_train, y_train)
            test_metrics = calculate_metrics(y_test, model.predict(X_test))
            train_r2 = r2_score(y_train, model.predict(X_train))

            rows.append({
                "grid_size_m": cell_size, "validation": "random_5fold",
                "model": model_name, "fold": fold_number, "spatial_block": np.nan,
                "n_train": len(train_index), "n_test": len(test_index),
                **test_metrics, "train_R2": train_r2,
            })

    return pd.DataFrame(rows)


def evaluate_spatial_kmeans(cell_size, matrix):
    X = matrix[BASELINE_FEATURES].copy()
    y = matrix[TARGET_COLUMN].copy()

    block_ids = assign_kmeans_blocks(matrix)
    unique_blocks = sorted(block_ids.dropna().unique())

    rows, skipped_blocks = [], []

    for model_name, model_template in make_models().items():
        print(f"{cell_size} m | spatial kmeans | {model_name}")
        fold_number = 0

        for block_id in unique_blocks:
            test_mask = block_ids == block_id
            train_mask = ~test_mask & block_ids.notna()

            test_index = np.flatnonzero(test_mask.to_numpy())
            train_index = np.flatnonzero(train_mask.to_numpy())

            if len(test_index) < MIN_SPATIAL_TEST_SAMPLES:
                skipped_blocks.append({
                    "grid_size_m": cell_size, "model": model_name,
                    "spatial_block": int(block_id),
                    "reason": "fewer_than_minimum_test_samples",
                    "n_test": len(test_index),
                })
                continue

            y_test = y.iloc[test_index]
            if y_test.nunique() < 2:
                skipped_blocks.append({
                    "grid_size_m": cell_size, "model": model_name,
                    "spatial_block": int(block_id),
                    "reason": "constant_test_target", "n_test": len(test_index),
                })
                continue

            fold_number += 1
            model = clone(model_template)
            X_train, X_test = X.iloc[train_index], X.iloc[test_index]
            y_train = y.iloc[train_index]

            model.fit(X_train, y_train)
            test_metrics = calculate_metrics(y_test, model.predict(X_test))
            train_r2 = r2_score(y_train, model.predict(X_train))

            rows.append({
                "grid_size_m": cell_size, "validation": "spatial_kmeans_leave_one_out",
                "model": model_name, "fold": fold_number, "spatial_block": int(block_id),
                "n_train": len(train_index), "n_test": len(test_index),
                **test_metrics, "train_R2": train_r2,
            })

    return pd.DataFrame(rows), pd.DataFrame(skipped_blocks)

## 10. Direct grid-size and preliminary model comparison

The tables below report Elastic Net, Random Forest and XGBoost separately under both random and spatial cross-validation.

Model-averaged performance across the three model families is also reported as supplementary screening evidence. Grid resolution is not selected from the averaged performance alone, but from the combined evidence on predictive performance, sample size, electricity-observation support and feature sparsity.

In [ ]:
all_fold_results = []
all_skipped_spatial_blocks = []

for grid_size in GRID_SIZES:
    matrix = matrices_by_size[grid_size]

    missing_features = [c for c in BASELINE_FEATURES if c not in matrix.columns]
    if missing_features:
        raise KeyError(f"{grid_size} m matrix is missing: {missing_features}")

    random_results = evaluate_random_5fold(grid_size, matrix)
    spatial_results, skipped_blocks = evaluate_spatial_kmeans(grid_size, matrix)

    all_fold_results.extend([random_results, spatial_results])
    if not skipped_blocks.empty:
        all_skipped_spatial_blocks.append(skipped_blocks)

fold_results = pd.concat(all_fold_results, ignore_index=True)

skipped_spatial_blocks = (
    pd.concat(all_skipped_spatial_blocks, ignore_index=True)
    if all_skipped_spatial_blocks
    else pd.DataFrame(columns=["grid_size_m", "model", "spatial_block", "reason", "n_test"])
)

model_summary = summarise_fold_results(fold_results)

fold_results_path = RESULT_OUTPUT_DIR / "grid_model_fold_results.csv"
model_summary_path = RESULT_OUTPUT_DIR / "grid_model_summary.csv"
skipped_blocks_path = RESULT_OUTPUT_DIR / "skipped_spatial_blocks.csv"

fold_results.to_csv(fold_results_path, index=False)
model_summary.to_csv(model_summary_path, index=False)
skipped_spatial_blocks.to_csv(skipped_blocks_path, index=False)

print("Saved:", fold_results_path)
print("Saved:", model_summary_path)
print("Saved:", skipped_blocks_path)

display(model_summary.sort_values(["validation", "R2_mean"], ascending=[True, False]).round(4))

## 10. Direct grid-size comparison

The tables below keep Random Forest and XGBoost separate, then provide averages across the two models.  
The model averages are useful for selecting a spatial resolution that is not dependent on one algorithm alone.

In [ ]:
random_summary = (
    model_summary[
        model_summary["validation"] == "random_5fold"
    ]
    .copy()
    .sort_values(["model", "R2_mean"], ascending=[True, False])
)

spatial_summary = (
    model_summary[
        model_summary["validation"]
        == "spatial_kmeans_leave_one_out"
    ]
    .copy()
    .sort_values(["model", "R2_mean"], ascending=[True, False])
)

print("Random 5-fold results")
display(random_summary.round(4))

print("Fixed k-means spatial CV results")
display(spatial_summary.round(4))


In [ ]:
model_average_performance = (
    model_summary
    .groupby(["grid_size_m", "validation"], as_index=False)
    .agg(
        model_average_R2=("R2_mean", "mean"),
        model_average_RMSE=("RMSE_mean", "mean"),
        model_average_MAE=("MAE_mean", "mean"),
        model_average_R2_std=("R2_std", "mean"),
        model_average_train_R2=("train_R2_mean", "mean"),
    )
)

performance_wide = model_average_performance.pivot(
    index="grid_size_m",
    columns="validation",
    values=[
        "model_average_R2",
        "model_average_RMSE",
        "model_average_MAE",
        "model_average_R2_std",
        "model_average_train_R2",
    ],
)

performance_wide.columns = [
    f"{metric}_{validation}"
    for metric, validation in performance_wide.columns
]
performance_wide = performance_wide.reset_index()

selection_evidence = performance_wide.merge(
    richness_summary,
    on="grid_size_m",
    how="left",
)

selection_evidence_path = (
    RESULT_OUTPUT_DIR / "grid_size_selection_evidence.csv"
)
selection_evidence.to_csv(selection_evidence_path, index=False)

print("Saved:", selection_evidence_path)
display(selection_evidence.round(4))


In [ ]:
# Model-performance plots
for model_name in [
    "ElasticNet",
    "RandomForest",
    "XGBoost",
]:
    model_plot = model_summary[
        model_summary["model"] == model_name
    ].copy()

    plt.figure(figsize=(8, 5))

    for validation_name in [
        "random_5fold",
        "spatial_kmeans_leave_one_out",
    ]:
        subset = (
            model_plot[
                model_plot["validation"] == validation_name
            ]
            .sort_values("grid_size_m")
        )

        plt.errorbar(
            subset["grid_size_m"],
            subset["R2_mean"],
            yerr=subset["R2_std"],
            marker="o",
            capsize=4,
            label=validation_name,
        )

    plt.axhline(0, linewidth=1)
    plt.xlabel("Grid size (m)")
    plt.ylabel("Mean R²")
    plt.title(f"{model_name}: grid-size comparison")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()


In [ ]:
# ── Diagnostic A: per-fold spatial R² detail ─────────────────────────────────
spatial_fold_detail = fold_results[
    fold_results["validation"] == "spatial_kmeans_leave_one_out"
].copy()

spatial_fold_detail = spatial_fold_detail.sort_values(
    ["grid_size_m", "model", "spatial_block"]
)

display(
    spatial_fold_detail[[
        "grid_size_m", "model", "spatial_block",
        "n_train", "n_test", "R2", "RMSE", "MAE", "train_R2",
    ]].round(3)
)

# Pivot for a compact view: rows = (grid_size, spatial_block), columns = model
r2_pivot = spatial_fold_detail.pivot_table(
    index=["grid_size_m", "spatial_block"],
    columns="model",
    values="R2",
)
print("\nR² by spatial block (rows) × model (columns):")
display(r2_pivot.round(3))

In [ ]:
# ── Diagnostic B: target distribution per spatial cluster ───────────────────
def cluster_target_summary(cell_size, matrix):
    block_ids = assign_kmeans_blocks(matrix)
    df = matrix[[TARGET_COLUMN]].copy()
    df["spatial_block"] = block_ids

    summary = (
        df.groupby("spatial_block")[TARGET_COLUMN]
        .agg(["count", "mean", "std", "median", "min", "max"])
        .reset_index()
    )
    summary["grid_size_m"] = cell_size
    summary["cv"] = summary["std"] / summary["mean"]
    return summary


cluster_target_all = pd.concat(
    [cluster_target_summary(gs, matrices_by_size[gs]) for gs in GRID_SIZES],
    ignore_index=True,
)

cluster_target_all = cluster_target_all[[
    "grid_size_m", "spatial_block", "count",
    "mean", "std", "cv", "median", "min", "max",
]].sort_values(["grid_size_m", "spatial_block"])

print("Target (elec_consumption) distribution by spatial cluster:")
display(cluster_target_all.round(2))

# Quick check: how different are cluster means from each other, per resolution?
mean_spread = (
    cluster_target_all
    .groupby("grid_size_m")["mean"]
    .agg(cluster_mean_min="min", cluster_mean_max="max")
)
mean_spread["relative_spread"] = (
    (mean_spread["cluster_mean_max"] - mean_spread["cluster_mean_min"])
    / mean_spread["cluster_mean_min"]
)
print("\nHow much do cluster-level target means differ within each resolution:")
display(mean_spread.round(3))

In [ ]:
# ── Diagnostic C: boxplot of target by cluster, per resolution ──────────────
fig, axes = plt.subplots(1, len(GRID_SIZES), figsize=(5 * len(GRID_SIZES), 5), sharey=True)

for ax, grid_size in zip(axes, sorted(GRID_SIZES, reverse=True)):
    matrix = matrices_by_size[grid_size].copy()
    matrix["spatial_block"] = assign_kmeans_blocks(matrix).astype(int)

    box_data = [
        matrix.loc[matrix["spatial_block"] == i, TARGET_COLUMN].values
        for i in range(N_SPATIAL_CLUSTERS)
    ]

    bp = ax.boxplot(
        box_data,
        patch_artist=True,
        labels=[f"C{i}" for i in range(N_SPATIAL_CLUSTERS)],
        showfliers=False,
    )
    for patch, color in zip(bp["boxes"], SPATIAL_PALETTE):
        patch.set_facecolor(color)

    ax.set_title(f"{grid_size} m")
    ax.set_xlabel("Spatial cluster")
    if grid_size == max(GRID_SIZES):
        ax.set_ylabel("elec_consumption")

fig.suptitle("Target distribution by KMeans spatial cluster", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Compare the three screening models at the 500 m resolution
# ============================================================

MODEL_ORDER = ["ElasticNet", "RandomForest", "XGBoost"]

comparison_500m = (
    model_summary[
        (model_summary["grid_size_m"] == 500)
        & (model_summary["model"].isin(MODEL_ORDER))
    ]
    .copy()
)

comparison_500m["model"] = pd.Categorical(
    comparison_500m["model"],
    categories=MODEL_ORDER,
    ordered=True,
)

comparison_500m = (
    comparison_500m
    .sort_values(["validation", "model"])
    .reset_index(drop=True)
)

# Keep the main performance metrics
comparison_500m_display = comparison_500m[
    [
        "validation",
        "model",
        "n_folds",
        "R2_mean",
        "R2_std",
        "RMSE_mean",
        "RMSE_std",
        "MAE_mean",
        "MAE_std",
        "train_R2_mean",
    ]
]

print("500 m model comparison")
display(comparison_500m_display.round(4))

# Save the comparison table
comparison_path = (
    RESULT_OUTPUT_DIR / "500m_three_model_comparison.csv"
)

comparison_500m_display.to_csv(
    comparison_path,
    index=False,
)

print("Saved:", comparison_path)

## 11. Transparent screening ranks

The rank below is only a **screening aid**, not an automatic scientific decision. It gives equal importance to:

1. higher average Random-CV R²;
2. higher average spatial-CV R²;
3. lower average validation instability;
4. fewer labelled grids containing fewer than five electricity points.

The final choice should also inspect the full evidence table, target variability, spatial-block diagnostics, and whether performance differences are practically meaningful.

In [ ]:
screening_table = selection_evidence.copy()

random_r2_col = "model_average_R2_random_5fold"
spatial_r2_col = (
    "model_average_R2_spatial_kmeans_leave_one_out"
)
random_std_col = "model_average_R2_std_random_5fold"
spatial_std_col = (
    "model_average_R2_std_spatial_kmeans_leave_one_out"
)

screening_table["rank_random_R2"] = screening_table[
    random_r2_col
].rank(ascending=False, method="min")

screening_table["rank_spatial_R2"] = screening_table[
    spatial_r2_col
].rank(ascending=False, method="min")

screening_table["mean_validation_R2_std"] = screening_table[
    [random_std_col, spatial_std_col]
].mean(axis=1)

screening_table["rank_stability"] = screening_table[
    "mean_validation_R2_std"
].rank(ascending=True, method="min")

screening_table["rank_label_reliability"] = screening_table[
    "elec_points_lt5_pct"
].rank(ascending=True, method="min")

screening_table["screening_mean_rank"] = screening_table[[
    "rank_random_R2",
    "rank_spatial_R2",
    "rank_stability",
    "rank_label_reliability",
]].mean(axis=1)

screening_table = screening_table.sort_values(
    ["screening_mean_rank", "grid_size_m"]
).reset_index(drop=True)

screening_columns = [
    "grid_size_m",
    random_r2_col,
    spatial_r2_col,
    "mean_validation_R2_std",
    "elec_points_lt5_pct",
    "poi_total_zero_pct",
    "target_cv",
    "screening_mean_rank",
]

screening_rank_path = (
    RESULT_OUTPUT_DIR / "grid_size_screening_rank.csv"
)
screening_table.to_csv(screening_rank_path, index=False)

print("Saved:", screening_rank_path)
display(screening_table[screening_columns].round(4))

